# example_pipeline_smoke_test — FabricOps pipeline smoke test example

This is a tiny runnable example, not a production workflow template. Use it after configuring `00_env_config` to validate that FabricOps can create metadata evidence, run guardrails, write lineage and runtime summaries, and publish a small target table in a Fabric workspace.

Safety notes:

- Uses only an in-memory Spark DataFrame.
- Uses overwrite mode only for the smoke target table.
- Metadata evidence tables remain append-only through the FabricOps evidence helpers.
- Writes a single overwrite target named `fabricops_smoke_target`.
- Prefixes smoke-test dataset/table names with `fabricops_smoke`.
- Does not read sample CSV, Excel, parquet, or production data.


## 1. Run `00_env_config`

Run the shared `00_env_config` notebook from the same `templates/notebooks` folder before executing the smoke-test cells.


In [ ]:
%run 00_env_config


## 2. Import required functions

These are the same thin FabricOps helpers used by `02_pipeline` for source/target preparation, guardrail orchestration, target writes, lineage, and runtime summaries.


In [ ]:
from datetime import datetime, timezone

from pyspark.sql import functions as F

from fabricops_kit import (
    guardrail_summary,
    prepare_source_table_configs,
    prepare_target_table_configs,
    run_table_guardrails,
    stop_if_any_guardrail_failed,
    write_pipeline_lineage,
    write_pipeline_run_summary,
    write_target_tables,
)


## 3. Capture smoke-test run context

This example avoids production data-agreement selection. It still passes public-safe smoke identifiers into evidence helpers so metadata rows are easy to recognize and clean up.


In [ ]:
PIPELINE_STARTED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
RUN_ID = RUN_CONTEXT.run_id
ENV_NAME = ENV
PIPELINE_NAME = "fabricops_smoke_pipeline"
DATASET_NAME = "fabricops_smoke_dataset"
AGREEMENT_ID = "fabricops_smoke_agreement"
AGREEMENT_CONTRACT_VERSION = "fabricops_smoke_v1"
NOTEBOOK_REGISTRY_ID = "fabricops_smoke_notebook_registry"
NOTEBOOK_ID = RUN_CONTEXT.runtime_metadata.get("currentNotebookId", "fabricops_smoke_notebook")


## 4. Source DataFrame/config block

Create a tiny in-memory source DataFrame. The variables mirror the config-driven pattern from `02_pipeline`, while `df` keeps the smoke test independent of external files or production tables.


In [ ]:
SOURCE_01_KEY = "fabricops_smoke_source_01"
SOURCE_01_DATASET_NAME = DATASET_NAME
SOURCE_01_TABLE_NAME = "fabricops_smoke_source"
SOURCE_01_STAGE = "source"

# Tiny in-memory Spark DataFrame: no external files or production tables.
df_source_01 = spark.createDataFrame(
    [
        (1, "2026-01-01", "new", 12.50, "US"),
        (2, "2026-01-02", "complete", 125.00, "GB"),
        (3, "2026-01-03", "complete", 42.00, "NL"),
    ],
    "customer_id long, business_date string, status string, amount double, country_code string",
)

DEFAULT_SOURCE_GUARDRAILS = {
    "schema_preset": "strict",
    "data_behavior": "fixed",
    "stability_check_type": "full_profile_hash",
    "dq_preset": "skip",
    "distribution_columns": ["status", "amount", "country_code"],
    "exclude_columns": None,
}

SOURCE_TABLES = [
    {
        "key": SOURCE_01_KEY,
        "df": df_source_01,
        "dataset_name": SOURCE_01_DATASET_NAME,
        "table_name": SOURCE_01_TABLE_NAME,
        "layer": SOURCE_01_STAGE,
        "stage": SOURCE_01_STAGE,
        "watermark_column": None,
        "watermark_value": None,
        "expected_schema": {
            "customer_id": "bigint",
            "business_date": "string",
            "status": "string",
            "amount": "double",
            "country_code": "string",
        },
    }
]


## 5. Prepare source configs

FabricOps applies source guardrail defaults and builds `SOURCE_CONFIG_BY_KEY`. The smoke source already supplies an in-memory DataFrame, so no external read occurs.


In [ ]:
SOURCE_TABLES, SOURCE_CONFIG_BY_KEY = prepare_source_table_configs(
    SOURCE_TABLES,
    DEFAULT_SOURCE_GUARDRAILS,
    CONFIG,
    ENV_NAME,
    spark,
)
SOURCE_01_CONFIG = SOURCE_CONFIG_BY_KEY[SOURCE_01_KEY]
df_source_01 = SOURCE_01_CONFIG["df"]


## 6. Guardrail orchestration is handled by FabricOps

Reusable guardrail orchestration lives in the FabricOps package. This example calls the same package helpers as `02_pipeline` instead of defining notebook-local framework code.


In [ ]:
# No user edits needed here. The package helpers imported above handle profiling,
# schema validation, stability checks, DQ checks, catalogue evidence, and stopping.


## 7. Run source guardrails before transformation

Source guardrails run before transformation. The smoke source skips approved-rule DQ lookup so a fresh workspace without reviewed DQ rules can still validate the rest of the orchestration path.


In [ ]:
source_guardrail_results = run_table_guardrails(
    SOURCE_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)

display(guardrail_summary(source_guardrail_results))
stop_if_any_guardrail_failed(source_guardrail_results)

source_schema_results = source_guardrail_results["schema_results"]
source_stability_results = source_guardrail_results["stability_results"]
source_dq_results = source_guardrail_results["dq_results"]
source_catalogue_status = source_guardrail_results["catalogue_status"]
source_evidence_definitions = source_guardrail_results["evidence_definitions"]


## 8. Transform source to target

Keep the transformation intentionally small: add an `amount_band` column from the in-memory source rows.


In [ ]:
df_target_01 = (
    df_source_01
    .withColumn(
        "amount_band",
        F.when(F.col("amount") >= F.lit(100), F.lit("high"))
        .when(F.col("amount") >= F.lit(25), F.lit("medium"))
        .otherwise(F.lit("low")),
    )
)


## 9. Target DataFrame/config block

Configure the target write to overwrite the smoke table `fabricops_smoke_target`. FabricOps adds audit columns during target preparation.


In [ ]:
TARGET_01_KEY = "fabricops_smoke_target_01"
TARGET_01_DATASET_NAME = DATASET_NAME
TARGET_01_TABLE_NAME = "fabricops_smoke_target"
TARGET_01_LAYER = "unified"
TARGET_01_KIND = "lakehouse"
TARGET_01_WRITE_MODE = "overwrite"

DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS = {
    "schema_preset": "strict",
    "data_behavior": "fixed",
    "stability_check_type": "full_profile_hash",
    "dq_preset": "skip",
    "distribution_columns": ["status", "amount", "amount_band", "country_code"],
    "exclude_columns": None,
    "target_kind": TARGET_01_KIND,
    "write_mode": TARGET_01_WRITE_MODE,
    "partition_by": None,
    "repartition_by": None,
    "overwrite_schema": True,
}

TARGET_TABLES = [
    {
        "key": TARGET_01_KEY,
        "df": df_target_01,
        "dataset_name": TARGET_01_DATASET_NAME,
        "layer": TARGET_01_LAYER,
        "table_name": TARGET_01_TABLE_NAME,
        "target_name": TARGET_01_TABLE_NAME,
        "target_layer": TARGET_01_LAYER,
        "target_kind": TARGET_01_KIND,
        "write_mode": TARGET_01_WRITE_MODE,
        "watermark_column": None,
        "watermark_value": None,
        "expected_schema": {
            "customer_id": "bigint",
            "business_date": "string",
            "status": "string",
            "amount": "double",
            "country_code": "string",
            "amount_band": "string",
            "_fabricops_run_id": "string",
            "_fabricops_pipeline_name": "string",
            "_fabricops_created_at": "string",
        },
    }
]


## 10. Prepare target configs

FabricOps applies target guardrail/write defaults, adds audit columns, and builds `TARGET_CONFIG_BY_KEY`.


In [ ]:
TARGET_TABLES, TARGET_CONFIG_BY_KEY = prepare_target_table_configs(
    TARGET_TABLES,
    DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS,
    RUN_ID,
    PIPELINE_NAME,
)
TARGET_01_CONFIG = TARGET_CONFIG_BY_KEY[TARGET_01_KEY]
TARGET_01_TABLE_NAME = TARGET_01_CONFIG["target_name"]
TARGET_01_WRITE_MODE = TARGET_01_CONFIG["write_mode"]


## 11. Run target guardrails before writing

Target writes happen only after all target guardrails pass.


In [ ]:
target_guardrail_results = run_table_guardrails(
    TARGET_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)

display(guardrail_summary(target_guardrail_results))
stop_if_any_guardrail_failed(target_guardrail_results)

target_schema_results = target_guardrail_results["schema_results"]
target_stability_results = target_guardrail_results["stability_results"]
target_dq_results = target_guardrail_results["dq_results"]
target_catalogue_status = target_guardrail_results["catalogue_status"]
target_evidence_definitions = target_guardrail_results["evidence_definitions"]


## 12. Write the smoke target

Write only the configured smoke target, using overwrite mode, after target guardrails have passed.


In [ ]:
target_write_status = write_target_tables(TARGET_TABLES, CONFIG, ENV_NAME)


## 13. Capture lineage

Record source-to-target lineage for the smoke example.


In [ ]:
LINEAGE_RELATIONSHIPS = [
    {
        "sources": [SOURCE_01_KEY],
        "targets": [TARGET_01_KEY],
        "operation": f"derive amount band and publish {TARGET_01_TABLE_NAME}",
        "description": f"{SOURCE_01_TABLE_NAME} in-memory rows are transformed into {TARGET_01_TABLE_NAME}.",
    },
]

lineage_result = write_pipeline_lineage(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    relationships=LINEAGE_RELATIONSHIPS,
    dataset_name=TARGET_01_CONFIG["dataset_name"],
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)


## 14. Capture runtime summary and finish

Record the runtime summary, display the final status, and print the required pipeline PASS message.


In [ ]:
catalogue_status = "written" if source_catalogue_status and target_catalogue_status else "not_written"

run_summary = write_pipeline_run_summary(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    started_at=PIPELINE_STARTED_AT,
    completed_at=datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    status="completed",
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    source_schema_results=source_schema_results,
    target_schema_results=target_schema_results,
    source_stability_results=source_stability_results,
    target_stability_results=target_stability_results,
    source_dq_results=source_dq_results,
    target_dq_results=target_dq_results,
    lineage_status=lineage_result.get("status", "unknown"),
    catalogue_status=catalogue_status,
    message="FabricOps pipeline smoke test completed and metadata evidence was written.",
)

display(
    {
        "target_write_status": target_write_status,
        "lineage_result": lineage_result,
        "run_summary": run_summary,
    }
)
print("PASS: FabricOps pipeline smoke test completed.")
